# Selección, negocio, Registry y reproducción

Preparado sin salidas hasta que finalicen los controles, la selección y la reproducción. Los dos finalistas y los controles estáticos se congelan por CV antes de la evaluación final. 2014J fue observado en v1: se declara benchmark retrospectivo reutilizado, no prueba inédita. Consultar resultados guardados no es una nueva selección sobre prueba.


In [1]:
from pathlib import Path
import os, sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
here = Path.cwd().resolve()
ROOT = next((p for p in [here, *here.parents] if (p/'config/protocolo-v2.json').is_file()), None)
if ROOT is None and (here/'pset_3/config/protocolo-v2.json').is_file(): ROOT=here/'pset_3'
assert ROOT is not None, 'Ejecutar desde pset_3 o notebooks; revisar README'
os.environ['PSET3_V2_ROOT'] = str(ROOT)
sys.path.insert(0, str(ROOT))
from src.seguimiento import protocolo
from src.common import sha256
plan = protocolo()
def report(relative):
    path = ROOT/relative
    if not path.is_file():
        raise RuntimeError('Etapa pendiente: falta '+relative+'. Revisar docs/estado-v2.json y esperar al coordinador; no sustituir por resultados de v1.')
    return json.loads(path.read_text())
print('Protocolo verificado:', plan['id'])
print('Este notebook consulta resultados reales y reproduce sus cálculos; no inicia nuevas búsquedas ni llamadas de inferencia API.')


Protocolo verificado: 07111fdf724a0cd8936ea5abc0ff983769f15d0f6216b24172c56aaf18e8ab0d
Este notebook consulta resultados reales y reproduce sus cálculos; no inicia nuevas búsquedas ni llamadas de inferencia API.


In [2]:
decision=report('docs/negocio/finalistas.json')
from src.common import identity
assert decision['decision_id']==identity(decision['content'])
d=decision['content']
assert d['protocol_id']==plan['id'] and d['v2_holdout_evaluated'] is False
display(pd.DataFrame([{'rol':role,'familia':d[role]['family'],'run_id':d[role]['run_id'],'cobertura_CV':d[role]['cv_coverage_mean'],'DE_CV':d[role]['cv_coverage_std'],'AP_CV':d[role]['cv_ap_fail_mean']} for role in ['winner','challenger']]))
print('Decision ID:',decision['decision_id'])
print('Diferencia:',d['winner_challenger_difference'],'| variación:',d['fold_variation_reference'])
print('Supera variación entre folds:',d['difference_exceeds_fold_variation'])
display(pd.DataFrame(d['tuning_improvements']))


,rol,familia,run_id,cobertura_CV,DE_CV,AP_CV
0,winner,boosting,dc2fa9bb8bee4944bd00611a6ac5c34f,0.233982,0.009338,0.583027
1,challenger,lineal,bb2b400975994efdb01ea418118dda67,0.232359,0.005617,0.556657


Decision ID: 2aa29e8d53d88d39bb331285faaa376ec556fe764ea13ef50651d045488ae2e4
Diferencia: 0.0016225800505769716 | variación: 0.009338351412435388
Supera variación entre folds: False


,family,default_run_id,tuned_run_id,delta_coverage,cv_spread_reference,improvement_exceeds_fold_variation,interpretation
0,lineal,7f75bbbab6ed4f56b58caa0850529b34,bb2b400975994efdb01ea418118dda67,0.003134,0.008012,False,sin mejora demostrada frente a la variación en...
1,boosting,61ef18564a504ec4bc39e454cbe1f8e9,dc2fa9bb8bee4944bd00611a6ac5c34f,0.005908,0.009338,False,sin mejora demostrada frente a la variación en...
2,bayes,dccc18435ccf489794cadd3ce4903342,e7c7aeb2d49b4f14bdbcbe39b30197c7,0.000422,0.010680,False,sin mejora demostrada frente a la variación en...


In [3]:
final=report('docs/seleccion/evaluacion.json')
assert final['status']=='registered' and final['decision_id']==decision['decision_id']
assert final['winner_changed_using_test'] is False and final['test_previously_observed_in_v1'] is True
rows=[]
for name,r in final['models'].items():
    rows.append({'modelo':name,'familia':r['family'],**r['technical'],**r['business']['0.1'],
                 'inferencia_prueba_seg':r['test_inference_seconds'],'API_prueba_USD':r['test_api_cost_usd']})
display(pd.DataFrame(rows))
display(pd.DataFrame([final['longitudinal_vs_matched_static']]))
display(pd.DataFrame([final['reactive']]))
display(pd.DataFrame(final['registry']))
print('Run de evaluación final:',final['final_run_id'])


,modelo,familia,ap_fail,logloss,brier_fail,tp,fp,fn,n,fail,k,coverage,precision_at_k,unique_students_selected,inferencia_prueba_seg,API_prueba_USD
0,winner,boosting,0.426507,0.842232,0.159718,465,407,1482,8758,1947,872,0.238829,0.533257,845,0.556767,0.0
1,challenger,lineal,0.438755,0.935294,0.159743,477,395,1470,8758,1947,872,0.244992,0.547018,846,0.077140,0.0
2,static_boosting,boosting,0.310455,0.925204,0.171608,309,563,1638,8758,1947,872,0.158706,0.354358,857,1.005651,0.0
3,static_lineal,lineal,0.315410,0.916969,0.170393,323,549,1624,8758,1947,872,0.165896,0.370413,854,0.237018,0.0
4,dummy,dummy,0.222311,13.889853,0.222311,174,698,1773,8758,1947,872,0.089368,0.199541,860,0.251228,0.0


,static_tp,longitudinal_tp,absolute_tp_gain,relative_tp_gain_pct,causal_effect
0,309,465,156,50.485437,False


,preventive_coverage,after_closure_tp,capacity,fail,after_closure_coverage
0,0.0,872,872,1947,0.447869


,alias,version,run_id,source_model_run_id,family,uri
0,champion,1,fcd23ae22d6c46ffb41c1e4f23e1c02d,dc2fa9bb8bee4944bd00611a6ac5c34f,boosting,models:/oulad-apoyo-academico-v2-modelo@champion
1,challenger,2,fcd23ae22d6c46ffb41c1e4f23e1c02d,bb2b400975994efdb01ea418118dda67,lineal,models:/oulad-apoyo-academico-v2-modelo@challe...


Run de evaluación final: fcd23ae22d6c46ffb41c1e4f23e1c02d


## Umbral, costo e interpretación

El cupo se aplica por edición: ordenar P(Fail) de mayor a menor y usar la clave solo para desempatar. No se impone un umbral universal de 0,5. Los cortes observados y los escenarios de costos relativos FN/FP se reportan; no se inventan precios institucionales. La regla busca máxima captura esperada con cupo fijo ocupado y costos homogéneos.

Dummy y reactiva son las dos baselines. El modelo estático, ajustado independientemente dentro de la misma familia, es el control que mide el aporte longitudinal. La reactiva asigna apoyo después del cierre, con capacidad acotada, y su cobertura preventiva es cero. Las diferencias predictivas no estiman el efecto causal de la tutoría.


In [4]:
display(pd.read_csv(ROOT/'docs/seleccion/por_edicion.csv'))
display(pd.read_csv(ROOT/'docs/seleccion/costos.csv'))
display(pd.read_csv(ROOT/'docs/seleccion/explicabilidad.csv').sort_values('mean_coverage_drop',ascending=False).head(15))
display(pd.DataFrame([report('docs/seleccion/explicacion_local.json')]))


,model,edition,tp,fp,fn,n,fail,k,coverage,precision_at_k,unique_students_selected,score_cutoff,tie_rule
0,winner,AAA|2014J,15,18,29,338,44,33,0.340909,0.454545,33,0.444749,"enrollment_key ascending, never label"
1,winner,BBB|2014J,103,67,280,1700,383,170,0.268930,0.605882,170,0.589944,"enrollment_key ascending, never label"
2,winner,CCC|2014J,76,110,327,1868,403,186,0.188586,0.408602,186,0.571930,"enrollment_key ascending, never label"
3,winner,DDD|2014J,78,63,285,1413,363,141,0.214876,0.553191,141,0.555126,"enrollment_key ascending, never label"
4,winner,EEE|2014J,57,42,138,992,195,99,0.292308,0.575758,99,0.521810,"enrollment_key ascending, never label"
5,winner,FFF|2014J,96,82,295,1789,391,178,0.245524,0.539326,178,0.530027,"enrollment_key ascending, never label"
6,winner,GGG|2014J,40,25,128,658,168,65,0.238095,0.615385,65,0.576693,"enrollment_key ascending, never label"
7,challenger,AAA|2014J,11,22,33,338,44,33,0.250000,0.333333,33,0.373162,"enrollment_key ascending, never label"
8,challenger,BBB|2014J,98,72,285,1700,383,170,0.255875,0.576471,170,0.494614,"enrollment_key ascending, never label"
9,challenger,CCC|2014J,86,100,317,1868,403,186,0.213400,0.462366,186,0.495818,"enrollment_key ascending, never label"


,model,family,rows,first_pass_seconds,first_pass_api_cost_usd,median_replay_seconds,repeats,replay_uses_cache_for_llm,training_cv_seconds,refit_seconds
0,winner,boosting,1000,0.244276,0.0,0.045537,5,False,8.967061,1.802219
1,challenger,lineal,1000,0.041653,0.0,0.018763,5,False,1.387693,0.316691
2,static_boosting,boosting,1000,0.052356,0.0,0.025549,5,False,3.636321,0.595178
3,static_lineal,lineal,1000,0.078169,0.0,0.016787,5,False,1.970705,0.361260
4,dummy,dummy,1000,0.036603,0.0,0.014492,5,False,1.306833,0.374981


,model,feature,mean_coverage_drop,std_coverage_drop,development_rows,repeats,causal
32,challenger,recency_days,0.019481,0.006494,200,2,False
26,challenger,active_days_through_cutoff,0.019481,0.006494,200,2,False
29,challenger,active_days_current_14d,0.019481,0.006494,200,2,False
5,winner,clicks_through_cutoff,0.012987,0.000000,200,2,False
8,winner,clicks_current_14d,0.012987,0.025974,200,2,False
1,winner,studied_credits,0.006494,0.019481,200,2,False
6,winner,active_days_through_cutoff,0.006494,0.006494,200,2,False
21,challenger,studied_credits,0.006494,0.006494,200,2,False
28,challenger,clicks_current_14d,0.006494,0.006494,200,2,False
18,winner,age_band,0.006494,0.006494,200,2,False


,method,predicted_fail_score_highest,interpretation
0,global permutation only; no forest path available,0.812453,descriptive model behavior; no causal interpre...


## Reproducción en proceso limpio

Se carga champion desde Model Registry y se recalcula AP y cobertura. Las nuevas peticiones API están bloqueadas durante la predicción; un LLM debe reproducir desde sus respuestas archivadas. Esta verificación no modifica la elección ni abre una nueva búsqueda sobre prueba.


In [5]:
import subprocess
result=subprocess.run([sys.executable,str(ROOT/'scripts/reproducir.py')],cwd=ROOT,capture_output=True,text=True)
assert result.returncode==0, result.stdout+result.stderr
reproduced=report('docs/seleccion/reproduccion-champion.json')
assert reproduced['status']=='ok' and reproduced['fresh_process']
assert reproduced['ap_exact'] and reproduced['business_exact'] and reproduced['api_new_requests']==0
display(pd.DataFrame([reproduced]))


,status,decision_id,fresh_process,uri,api_new_requests,predictions_array_exact,prediction_max_abs_difference,ap_exact,business_exact,technical,business
0,ok,2aa29e8d53d88d39bb331285faaa376ec556fe764ea13e...,True,models:/oulad-apoyo-academico-v2-modelo@champion,0,True,0.0,True,True,"{'ap_fail': 0.4265069557086188, 'logloss': 0.8...","{'tp': 465, 'fp': 407, 'fn': 1482, 'n': 8758, ..."
